In [1]:
from pyspark.sql import SparkSession

spark = SparkSession. \
builder. \
appName("week10-lesson-1"). \
config("spark.sql.warehouse.dir", f"/user/itv027484/warehouse"). \
enableHiveSupport(). \
master('yarn'). \
getOrCreate()
from pyspark.sql.types import *
from pyspark.sql.functions import *

In [2]:
order_schema= 'order_id long, order_date string , customer_id long, order_status string'

In [3]:
orders_df = spark.read.format('csv').schema(order_schema).load('/public/trendytech/orders/orders_1gb.csv')

In [4]:
orders_df.show(4)

+--------+--------------------+-----------+---------------+
|order_id|          order_date|customer_id|   order_status|
+--------+--------------------+-----------+---------------+
|       1|2013-07-25 00:00:...|      11599|         CLOSED|
|       2|2013-07-25 00:00:...|        256|PENDING_PAYMENT|
|       3|2013-07-25 00:00:...|      12111|       COMPLETE|
|       4|2013-07-25 00:00:...|       8827|         CLOSED|
+--------+--------------------+-----------+---------------+
only showing top 4 rows



In [5]:
orders_df.groupby("order_status").count().write.format('csv').mode('overwrite').save('ordersgroupby')

### NOOP write

In [6]:
orders_df.groupby("order_status").count().write.format('noop').mode('overwrite').save('ordersgroupby')

### Broad cast join

In [7]:
cust_df = spark.read.format('csv').option('inferSchema','true').load('/public/trendytech/retail_db/customers/')

In [8]:
cust_df1 = cust_df.toDF("cust_id","cust_fname","cust_lname","cust_email","cust_pass","cust_addr1","cust_city","cust_state","cust_zip")

In [9]:
cust_df1.show(4)

+-------+----------+----------+----------+---------+--------------------+-----------+----------+--------+
|cust_id|cust_fname|cust_lname|cust_email|cust_pass|          cust_addr1|  cust_city|cust_state|cust_zip|
+-------+----------+----------+----------+---------+--------------------+-----------+----------+--------+
|      1|   Richard| Hernandez| XXXXXXXXX|XXXXXXXXX|  6303 Heather Plaza|Brownsville|        TX|   78521|
|      2|      Mary|   Barrett| XXXXXXXXX|XXXXXXXXX|9526 Noble Embers...|  Littleton|        CO|   80126|
|      3|       Ann|     Smith| XXXXXXXXX|XXXXXXXXX|3422 Blue Pioneer...|     Caguas|        PR|     725|
|      4|      Mary|     Jones| XXXXXXXXX|XXXXXXXXX|  8324 Little Common| San Marcos|        CA|   92069|
+-------+----------+----------+----------+---------+--------------------+-----------+----------+--------+
only showing top 4 rows



In [10]:
spark.conf.get('spark.sql.autoBroadcastJoinThreshold')

'10485760b'

In [11]:
10485760/(1024*1024)

10.0

###  disable broadcast join

In [12]:
spark.conf.set('spark.sql.autoBroadcastJoinThreshold','-1')

In [13]:
spark.conf.get('spark.sql.autoBroadcastJoinThreshold')

'-1'

In [14]:
orders_df.join(cust_df1,orders_df.customer_id == cust_df1.cust_id,"inner").write.format('noop').mode('overwrite').save()

###  enable broadcast join

In [15]:
spark.conf.set('spark.sql.autoBroadcastJoinThreshold','10485760b')

In [30]:
orders_df.join(cust_df1,orders_df.customer_id == cust_df1.cust_id,"inner").write.format('noop').mode('overwrite').save()

### left outer join

In [17]:
orders_df.join(cust_df1,orders_df.customer_id == cust_df1.cust_id,"left").write.format('noop').mode('overwrite').save()

#### right outer join --- cannot utlize broadcast join as the whole data from right table is needed for a join

In [19]:
orders_df.join(cust_df1,orders_df.customer_id == cust_df1.cust_id,"right").write.format('noop').mode('overwrite').save()

#### Forced broadcast- might fail if data doesnt fit in memory

In [20]:
cust_df1.join(broadcast(orders_df),cust_df1.cust_id == orders_df.customer_id,"left").write.format('noop').mode('overwrite').save()

Py4JJavaError: An error occurred while calling o127.save.
: java.util.concurrent.ExecutionException: org.apache.spark.util.SparkFatalException
	at java.util.concurrent.FutureTask.report(FutureTask.java:122)
	at java.util.concurrent.FutureTask.get(FutureTask.java:206)
	at org.apache.spark.sql.execution.exchange.BroadcastExchangeExec.doExecuteBroadcast(BroadcastExchangeExec.scala:194)
	at org.apache.spark.sql.execution.InputAdapter.doExecuteBroadcast(WholeStageCodegenExec.scala:515)
	at org.apache.spark.sql.execution.SparkPlan.$anonfun$executeBroadcast$1(SparkPlan.scala:193)
	at org.apache.spark.sql.execution.SparkPlan.$anonfun$executeQuery$1(SparkPlan.scala:218)
	at org.apache.spark.rdd.RDDOperationScope$.withScope(RDDOperationScope.scala:151)
	at org.apache.spark.sql.execution.SparkPlan.executeQuery(SparkPlan.scala:215)
	at org.apache.spark.sql.execution.SparkPlan.executeBroadcast(SparkPlan.scala:189)
	at org.apache.spark.sql.execution.joins.BroadcastHashJoinExec.prepareBroadcast(Broad

In [21]:
cust_df1.createOrReplaceTempView("customers")

In [22]:
orders_df.createOrReplaceTempView("orders")

In [31]:
spark.sql("select * from customers inner join orders on orders.customer_id = customers.cust_id").write.format('noop').mode('overwrite').save()